# TorGen: DETR-CVAE for Tornado Outbreak Generation

Install the package, mount Drive, and train.

In [ ]:
!pip install -q --no-cache-dir git+https://github.com/jcaramichaellehigh/TorGen.git

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import logging
logging.basicConfig(level=logging.INFO)

from torgen.training import TrainConfig, train

# Run D: Weather bypass + low KL weight to prioritize reconstruction
config = TrainConfig(
    drive_dir="/content/drive/MyDrive/raw",
    checkpoint_dir="/content/drive/MyDrive/checkpoints_d",
    exists_threshold=0.3,
    lambda_exists=15.0,
    kl_weight=0.1,
)

trainer = train(config)

In [ ]:
print(f"Epochs trained: {trainer.epoch}")
print(f"Final train loss: {trainer.loss_history['train_total'][-1]:.4f}")
print(f"Final val loss: {trainer.loss_history['val_total'][-1]:.4f}")
print(f"Best val loss: {trainer.best_val_loss:.4f}")

if trainer.loss_history["train_kl"]:
    print(f"Final KL: {trainer.loss_history['train_kl'][-1]:.4f}")
    print(f"Final recon: {trainer.loss_history['train_recon'][-1]:.4f}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
from torgen.viz.plots import plot_training_curves

fig = plot_training_curves(trainer.loss_history)
plt.show()

In [ ]:
import json
import os

eval_dir = os.path.join(config.checkpoint_dir, "eval")
summary_path = os.path.join(eval_dir, "summary.json")
if os.path.exists(summary_path):
    with open(summary_path) as f:
        results = json.load(f)
    print("Evaluation Results:")
    for k, v in results.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.4f}")
        else:
            print(f"  {k}: {v}")
else:
    print("No evaluation results found (evaluation runs automatically after training)")

## Diversity: Multiple Z Draws per Day

In [ ]:
import os
import torch
from torgen.data.dataset import _load_pt, coords_to_bearing_length
from torgen.viz.plots import _plot_tracks

threshold = config.exists_threshold

# Pick days spanning regimes — adjust after inspecting GT counts
days = ["2011-04-27.pt", "2019-05-20.pt", "2023-03-31.pt"]
n_samples = 8

n_rows = len(days)
n_cols = 1 + n_samples
fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3.5 * n_rows))
if n_rows == 1:
    axes = axes[None, :]

trainer.model.eval()

for row, day_file in enumerate(days):
    path = os.path.join(config.local_cache_dir, day_file)
    sample = _load_pt(path)
    gt_tracks = coords_to_bearing_length(sample["tracks"])
    date = sample.get("date", day_file.replace(".pt.gz", "").replace(".pt", ""))
    n_gt = gt_tracks.shape[0]

    _plot_tracks(axes[row, 0], gt_tracks, title=f"GT — {date}\n({n_gt} tracks)")

    wx = sample["wx"].unsqueeze(0).to(trainer.device)

    for col in range(n_samples):
        with torch.no_grad():
            out = trainer.model.generate(wx)

        preds = out["preds"]
        exists = preds["exists"][0].squeeze(-1)  # (Q,)
        mask = exists > threshold

        if mask.any():
            coords = preds["coords"][0][mask]
            bearing = preds["bearing"][0][mask]
            length = preds["length"][0][mask]
            width = preds["width"][0][mask]
            ef = preds["ef_logits"][0][mask].argmax(dim=-1).float()

            tracks_for_plot = torch.zeros(coords.shape[0], 6)
            tracks_for_plot[:, :2] = coords.cpu()
            tracks_for_plot[:, 2] = bearing.squeeze(-1).cpu()
            tracks_for_plot[:, 3] = length.squeeze(-1).cpu()
            tracks_for_plot[:, 4] = width.squeeze(-1).cpu()
            tracks_for_plot[:, 5] = ef.cpu()
        else:
            tracks_for_plot = torch.zeros(0, 6)

        n_pred = tracks_for_plot.shape[0]
        _plot_tracks(axes[row, 1 + col], tracks_for_plot,
                     title=f"Sample {col + 1}\n({n_pred} tracks)")

fig.tight_layout()
plt.show()

## Sample from Trained Model

Pick a day and generate multiple realizations from the same weather input.

In [ ]:
# Pick a day (already cached locally)
pt_path = os.path.join(config.local_cache_dir, "2011-04-27.pt")
sample = _load_pt(pt_path)
wx = sample["wx"].unsqueeze(0).to(trainer.device)
gt_tracks = coords_to_bearing_length(sample["tracks"])
date = sample.get("date", "2011-04-27")

n_samples = 5
trainer.model.eval()

n_cols = 1 + n_samples
fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))

# Ground truth
_plot_tracks(axes[0], gt_tracks, title=f"Ground Truth\n{date}")

# Sampled panels
for i in range(n_samples):
    with torch.no_grad():
        out = trainer.model.generate(wx)

    preds = out["preds"]
    exists = preds["exists"][0].squeeze(-1)  # (Q,)
    mask = exists > threshold

    if mask.any():
        coords = preds["coords"][0][mask]
        bearing = preds["bearing"][0][mask]
        length = preds["length"][0][mask]
        width = preds["width"][0][mask]
        ef = preds["ef_logits"][0][mask].argmax(dim=-1).float()

        tracks_for_plot = torch.zeros(coords.shape[0], 6)
        tracks_for_plot[:, :2] = coords.cpu()
        tracks_for_plot[:, 2] = bearing.squeeze(-1).cpu()
        tracks_for_plot[:, 3] = length.squeeze(-1).cpu()
        tracks_for_plot[:, 4] = width.squeeze(-1).cpu()
        tracks_for_plot[:, 5] = ef.cpu()
    else:
        tracks_for_plot = torch.zeros(0, 6)

    n_tracks = tracks_for_plot.shape[0]
    _plot_tracks(axes[1 + i], tracks_for_plot, title=f"Sample {i+1}\n({n_tracks} tracks)")

fig.suptitle(f"TorGen DETR-CVAE: {date}", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()